<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/13a_lstm_tuning_compacto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB13a — Tuning compacto da LSTM

## 1. Papel da etapa

O NB13a é uma etapa intermediária e complementar ao NB13. Seu objetivo é verificar se uma varredura compacta de hiperparâmetros da LSTM altera a decisão metodológica de manter o modelo tabular vencedor do NB11 como fonte preferencial de escores para o NB14.

O notebook pode consumir artefatos das etapas anteriores, inclusive do NB13, mas não sobrescreve arquivos produzidos pelo notebook pai. Todos os artefatos desta etapa usam o prefixo `13a_`.

## 2. Questão experimental

**Um tuning compacto da LSTM produz ganho temporal suficiente para substituir os escores do NB11 no NB14?**

A questão não reabre uma busca neural ampla. Ela testa, em escopo controlado, se a conclusão do NB13 permanece após variações moderadas da memória sequencial, da capacidade da camada recorrente, do dropout e da taxa de aprendizagem.

## 3. Hipótese avaliada

A hipótese experimental é que parte da diferença entre a LSTM do NB13 e a Regressão Logística do NB11 possa decorrer de configuração insuficiente da rede recorrente.

A hipótese somente recebe suporte para alterar o fluxo principal se a melhor configuração confirmada superar o F1 médio do vencedor tabular no `TimeSeriesSplit` pela margem mínima previamente definida.

## 4. Referências de comparação

A etapa lê as referências diretamente dos artefatos vigentes:

| Referência | Configuração | F1 médio TSCV |
|---|---|---:|
| NB11 | Regressão Logística, `temporal_core` | 0,5773 |
| NB13 | LSTM, `temporal_core`, 16 unidades | 0,5591 |

Esses valores funcionam como comparadores de entrada. O resultado do NB13a será calculado novamente a partir dos artefatos produzidos por esta execução.

## 5. Cenário e conjunto de atributos

O tuning é restrito ao ponto de comparação principal:

```text
Cenário: W5_K24_H12_P1_TRAIN_M2S
Conjunto de atributos: temporal_core
Protocolo principal: TimeSeriesSplit
Métrica de seleção: F1 médio
```

A restrição evita garimpagem experimental e concentra a análise na configuração mais relevante para a decisão do pipeline.

## 6. Contrato de atributos

O conjunto `raw_core` possui 6 atributos:

```text
fail_rate
n_events
n_failed
n_machines
n_collections
event_LOST_count
```

O conjunto `temporal_core` possui 13 atributos:

```text
fail_rate
n_events
n_failed
n_machines
n_collections
event_LOST_count
lag_1
lag_2
lag_3
rolling_mean_1h
rolling_std_1h
pct_change
zscore_expanding
```

`event_FAIL_count` permanece no dataframe sequencial exclusivamente para proveniência e auditoria. A coluna não integra os tensores da LSTM.

## 7. Preflight anterior ao treinamento

Antes da primeira avaliação, o notebook verifica:

- cardinalidades 6/13 e unicidade dos nomes;
- correspondência entre `11_feature_sets.json`, o schema do NB11 e o vetor ativo;
- ausência de `event_FAIL_count` nos preditores;
- igualdade entre `n_failed` e `event_FAIL_count`;
- consistência de `fail_rate = n_failed / n_events`;
- ausência de alvo, estado, elegibilidade, cenário, tempo absoluto e informação futura;
- inexistência de atributos exatamente duplicados no dataframe efetivamente imputado e utilizado para formar as sequências;
- dimensão final dos tensores igual a 13.

Qualquer não conformidade interrompe a execução antes do treinamento.

## 8. Grid compacto

O espaço de busca combina:

```text
sequence_length: 12, 24, 36
units: 8, 16, 32
dropout: 0,10; 0,20; 0,30
learning_rate: 0,001; 0,0005
```

O grid contém 54 configurações. As extensões de sequência correspondem a 60, 120 e 180 minutos na granularidade de 5 minutos.

## 9. Desenho em duas fases

A avaliação é organizada em:

1. **Triagem:** todas as 54 configurações, uma semente e cinco folds temporais.
2. **Confirmação:** as melhores configurações da triagem, três sementes e cinco folds temporais.
3. **Corte fixo complementar:** a configuração confirmada é reavaliada no corte temporal 80/20 com as três sementes.

O plano nominal corresponde a 270 avaliações na triagem, até 45 na confirmação e até 3 no corte fixo. Essas quantidades não são tratadas como resultado antecipado: o notebook registra separadamente as avaliações efetivamente produzidas, válidas e ignoradas.

## 10. Protocolo temporal

O `TimeSeriesSplit` constitui o protocolo principal de seleção. Em cada fold:

- o escalonador é ajustado somente no trecho de treino;
- a parte final do treino é reservada para validação interna e early stopping;
- a ordem temporal é preservada;
- `shuffle=False`;
- treino e teste precisam conter as duas classes.

O corte fixo 80/20 é complementar e não substitui a validação progressiva.

## 11. Treinamento e métrica

Cada rede utiliza:

- camada LSTM;
- camada de dropout;
- saída sigmoidal;
- perda `binary_crossentropy`;
- otimizador Adam;
- ponderação de classes calculada no treino;
- early stopping e redução adaptativa da taxa de aprendizagem.

A seleção é conduzida pelo F1 médio TSCV. Precisão, recall, ROC-AUC, Average Precision, Brier Score, épocas e tempos de ajuste permanecem como diagnósticos complementares.

## 12. Interpretação dos escores

As saídas sigmoidais são tratadas como **escores relativos de risco temporal**. Elas não são interpretadas como probabilidades operacionais calibradas.

O limiar de 0,50 padroniza a comparação experimental. A análise de limiares de alerta, custos e falsos alertas pertence ao NB14.

## 13. Critério de promoção

A LSTM ajustada somente pode ser indicada como candidata a substituir a fonte tabular quando:

```text
F1_tuned_LSTM − F1_NB11 ≥ 0,0300
```

Caso o critério não seja satisfeito, os escores do NB11 permanecem como entrada preferencial do NB14. Um ganho em relação à LSTM do NB13, isoladamente, não é suficiente para promoção.

## 14. Rastreabilidade e artefatos

A etapa produz:

```text
13a_feature_duplicate_audit.csv
13a_lstm_tuning_stage1_results.csv
13a_lstm_tuning_stage1_agg.csv
13a_lstm_tuning_stage2_results.csv
13a_lstm_tuning_stage2_agg.csv
13a_lstm_tuning_fixed_results.csv
13a_lstm_tuning_fixed_results_agg.csv
13a_lstm_tuning_scores.parquet
13a_lstm_tuning_best_config.json
13a_lstm_tuning_summary.json
13a_lstm_tuning_conclusion.txt
```

Artefatos `13a_` anteriores são preservados em backup e retirados da área ativa antes da nova materialização. Nenhum arquivo `13_` é alterado.

## 15. Política editorial da etapa

O NB13a deve registrar quatro elementos: hipótese, espaço de tuning, resultado e decisão de promoção ou não promoção.

Não é exigida figura própria. O diretório `figures_nb13a_lstm_tuning` permanece intencionalmente vazio; a evidência principal é tabular e estruturada.


In [ ]:
# ============================================================
# NB13a — Tuning compacto da LSTM
# ------------------------------------------------------------
# Notebook intermediário derivado do NB13.
#
# Regras:
#   - Pode ler artefatos anteriores, inclusive do NB13.
#   - Não sobrescreve artefatos do notebook pai.
#   - Todos os outputs usam prefixo 13a_.
#   - Usa temporal_core com 13 atributos.
#   - event_FAIL_count permanece apenas para proveniência.
#
# Objetivo:
#   Avaliar se um tuning compacto da LSTM altera a decisão
#   metodológica de manter os escores tabulares do NB11 no NB14.
# ============================================================

from __future__ import annotations

import os
import re
import json
import math
import time
import shutil
import random
import warnings
from pathlib import Path
from datetime import datetime
from itertools import product
from typing import Dict, List, Tuple, Any, Optional

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Ambiente e bibliotecas
# ------------------------------------------------------------

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"[AVISO] Não foi possível montar o Google Drive automaticamente: {exc}")

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

print("TensorFlow:", tf.__version__)

# ------------------------------------------------------------
# 2. Configuração geral
# ------------------------------------------------------------

RUN_MODE = "NB13A_COMPACT_TUNING"

# Diretórios principais
DEFAULT_REPORTS_PATH = Path("/content/drive/MyDrive/Mestrado/04-reports")
LOCAL_FALLBACK_REPORTS_PATH = Path.cwd()

REPORTS_PATH = (
    DEFAULT_REPORTS_PATH
    if DEFAULT_REPORTS_PATH.exists()
    else LOCAL_FALLBACK_REPORTS_PATH
)
FIGURES_PATH = REPORTS_PATH / "figures_nb13a_lstm_tuning"

# Entradas principais
SEQUENCE_INPUT_PATH = REPORTS_PATH / "11_sequence_input_features.parquet"
MODEL_INPUT_PATH = REPORTS_PATH / "11_model_input_features.parquet"
FEATURE_SETS_PATH = REPORTS_PATH / "11_feature_sets.json"
SCHEMA_PATH = REPORTS_PATH / "11_model_input_features_schema.json"
NB11_SUMMARY_PATH = REPORTS_PATH / "11_nb11_summary.json"
WINNER_MODEL_PATH = REPORTS_PATH / "11_winner_model.json"
NB13_SUMMARY_PATH = REPORTS_PATH / "13_lstm_summary.json"

# Saídas NB13a — nunca sobrescrever 13_*
OUT_FEATURE_AUDIT = REPORTS_PATH / "13a_feature_duplicate_audit.csv"
OUT_STAGE1_RESULTS = REPORTS_PATH / "13a_lstm_tuning_stage1_results.csv"
OUT_STAGE1_AGG = REPORTS_PATH / "13a_lstm_tuning_stage1_agg.csv"
OUT_STAGE2_RESULTS = REPORTS_PATH / "13a_lstm_tuning_stage2_results.csv"
OUT_STAGE2_AGG = REPORTS_PATH / "13a_lstm_tuning_stage2_agg.csv"
OUT_FIXED_RESULTS = REPORTS_PATH / "13a_lstm_tuning_fixed_results.csv"
OUT_FIXED_AGG = REPORTS_PATH / "13a_lstm_tuning_fixed_results_agg.csv"
OUT_SCORES = REPORTS_PATH / "13a_lstm_tuning_scores.parquet"
OUT_BEST_CONFIG = REPORTS_PATH / "13a_lstm_tuning_best_config.json"
OUT_SUMMARY = REPORTS_PATH / "13a_lstm_tuning_summary.json"
OUT_CONCLUSION = REPORTS_PATH / "13a_lstm_tuning_conclusion.txt"

# Contrato normativo de atributos
RAW_CORE = [
    "fail_rate",
    "n_events",
    "n_failed",
    "n_machines",
    "n_collections",
    "event_LOST_count",
]
TEMPORAL_CORE = RAW_CORE + [
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_1h",
    "rolling_std_1h",
    "pct_change",
    "zscore_expanding",
]
PROVENANCE_ONLY_COLUMNS = ["event_FAIL_count"]

assert len(RAW_CORE) == 6
assert len(TEMPORAL_CORE) == 13
assert len(RAW_CORE) == len(set(RAW_CORE))
assert len(TEMPORAL_CORE) == len(set(TEMPORAL_CORE))
assert "event_FAIL_count" not in RAW_CORE
assert "event_FAIL_count" not in TEMPORAL_CORE

# Escopo metodológico
MAIN_SCENARIO_DEFAULT = "W5_K24_H12_P1_TRAIN_M2S"
FEATURE_SET_TO_TUNE = "temporal_core"

# Grid compacto
SEQUENCE_LENGTH_GRID = [12, 24, 36]
LSTM_UNITS_GRID = [8, 16, 32]
DROPOUT_GRID = [0.10, 0.20, 0.30]
LEARNING_RATE_GRID = [0.001, 0.0005]

# Desenho em duas fases
STAGE1_SEEDS = [42]
STAGE2_SEEDS = [42, 123, 2026]
TOP_K_FOR_CONFIRMATION = 3

# Validação
TSCV_SPLITS = 5
FIXED_TRAIN_FRACTION = 0.80
VALIDATION_FRACTION_FROM_TRAIN = 0.20

# Treinamento
MAX_EPOCHS = 50
BATCH_SIZE = 128
EARLY_STOPPING_PATIENCE = 8
REDUCE_LR_PATIENCE = 4
CLASSIFICATION_THRESHOLD = 0.50

# Decisão
MIN_DELTA_F1_TO_FEED_NB14 = 0.0300
MIN_DELTA_F1_TO_CONSIDER_TUNING_GAIN = 0.0100

# Segurança operacional
BACKUP_PREVIOUS_13A_OUTPUTS = True
SAVE_KERAS_MODELS = False
KERAS_VERBOSE = 0


# ------------------------------------------------------------
# 3. Utilitários
# ------------------------------------------------------------

def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def json_default(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        if np.isnan(obj):
            return None
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (Path,)):
        return str(obj)
    if pd.isna(obj):
        return None
    return str(obj)

def load_json(path: Path, required: bool = True) -> Dict[str, Any]:
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Arquivo não encontrado: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj: Dict[str, Any], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=json_default)

def nb13a_output_paths() -> List[Path]:
    return [
        OUT_FEATURE_AUDIT,
        OUT_STAGE1_RESULTS,
        OUT_STAGE1_AGG,
        OUT_STAGE2_RESULTS,
        OUT_STAGE2_AGG,
        OUT_FIXED_RESULTS,
        OUT_FIXED_AGG,
        OUT_SCORES,
        OUT_BEST_CONFIG,
        OUT_SUMMARY,
        OUT_CONCLUSION,
    ]


def backup_and_clear_existing_13a_outputs() -> Tuple[Optional[Path], List[str]]:
    """
    Preserva os artefatos 13a existentes e remove-os da área ativa antes
    da nova materialização. Nenhum artefato 13_* é alterado.
    """
    existing_files = [p for p in nb13a_output_paths() if p.exists()]
    figure_entries = (
        list(FIGURES_PATH.iterdir())
        if FIGURES_PATH.exists()
        else []
    )

    if not existing_files and not figure_entries:
        FIGURES_PATH.mkdir(parents=True, exist_ok=True)
        return None, []

    backup_dir = None
    if BACKUP_PREVIOUS_13A_OUTPUTS:
        backup_dir = (
            REPORTS_PATH
            / "_backup_previous_artifacts"
            / f"nb13a_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        )
        backup_dir.mkdir(parents=True, exist_ok=True)

        for path in existing_files:
            shutil.copy2(path, backup_dir / path.name)

        if figure_entries:
            figures_backup = backup_dir / FIGURES_PATH.name
            figures_backup.mkdir(parents=True, exist_ok=True)
            for entry in figure_entries:
                destination = figures_backup / entry.name
                if entry.is_dir():
                    shutil.copytree(entry, destination)
                else:
                    shutil.copy2(entry, destination)

    removed = []
    for path in existing_files:
        path.unlink()
        removed.append(str(path))

    if FIGURES_PATH.exists():
        shutil.rmtree(FIGURES_PATH)
    FIGURES_PATH.mkdir(parents=True, exist_ok=True)

    return backup_dir, removed


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def safe_float(x) -> Optional[float]:
    try:
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None

def extract_feature_set(
    feature_sets_obj: Dict[str, Any],
    scenario_label: str,
    feature_set: str,
) -> List[str]:
    """
    Extrai o conjunto de atributos sem remover silenciosamente colunas.
    A lista retornada deve coincidir com o contrato normativo.
    """
    candidates: List[Any] = []

    candidates.append(feature_sets_obj.get(feature_set))

    by_scenario = feature_sets_obj.get("feature_sets_by_scenario", {})
    if isinstance(by_scenario, dict):
        scenario_node = by_scenario.get(scenario_label, {})
        if isinstance(scenario_node, dict):
            candidates.append(scenario_node.get(feature_set))

    legacy_container = feature_sets_obj.get("feature_sets", {})
    if isinstance(legacy_container, dict):
        candidates.append(legacy_container.get(feature_set))
        scenario_node = legacy_container.get(scenario_label, {})
        if isinstance(scenario_node, dict):
            candidates.append(scenario_node.get(feature_set))

    direct_scenario = feature_sets_obj.get(scenario_label, {})
    if isinstance(direct_scenario, dict):
        candidates.append(direct_scenario.get(feature_set))

    for candidate in candidates:
        if isinstance(candidate, list):
            return list(candidate)
        if isinstance(candidate, dict):
            features = candidate.get("features")
            if isinstance(features, list):
                return list(features)

    raise ValueError(
        f"Não foi possível extrair {feature_set} para {scenario_label} "
        "a partir de 11_feature_sets.json."
    )


def extract_schema_feature_set(
    schema_obj: Dict[str, Any],
    feature_set: str,
) -> List[str]:
    normative = schema_obj.get("normative_feature_sets", {})
    if not isinstance(normative, dict):
        return []

    candidate = normative.get(feature_set)
    if isinstance(candidate, list):
        return list(candidate)
    if isinstance(candidate, dict):
        features = candidate.get("features")
        if isinstance(features, list):
            return list(features)
    return []


def validate_feature_contract(
    feature_sets_obj: Dict[str, Any],
    schema_obj: Dict[str, Any],
    scenario_label: str,
    active_features: List[str],
) -> None:
    expected_cardinality = feature_sets_obj.get("expected_cardinality", {})

    raw_from_file = extract_feature_set(
        feature_sets_obj,
        scenario_label,
        "raw_core",
    )
    temporal_from_file = extract_feature_set(
        feature_sets_obj,
        scenario_label,
        "temporal_core",
    )

    assert raw_from_file == RAW_CORE, (
        "raw_core do artefato difere do contrato normativo."
    )
    assert temporal_from_file == TEMPORAL_CORE, (
        "temporal_core do artefato difere do contrato normativo."
    )
    assert active_features == TEMPORAL_CORE, (
        "O vetor ativo do tuning deve coincidir com temporal_core."
    )

    assert int(expected_cardinality.get("raw_core", 6)) == 6
    assert int(expected_cardinality.get("temporal_core", 13)) == 13

    schema_raw = extract_schema_feature_set(schema_obj, "raw_core")
    schema_temporal = extract_schema_feature_set(
        schema_obj,
        "temporal_core",
    )
    if schema_raw:
        assert schema_raw == RAW_CORE, (
            "raw_core do schema difere do contrato normativo."
        )
    if schema_temporal:
        assert schema_temporal == TEMPORAL_CORE, (
            "temporal_core do schema difere do contrato normativo."
        )

    assert len(active_features) == 13
    assert len(active_features) == len(set(active_features))
    assert "event_FAIL_count" not in active_features
    legacy_feature_set_name = "raw" + "_7"
    assert legacy_feature_set_name not in feature_sets_obj


def detect_schema_columns(schema: Dict[str, Any], df_columns: List[str]) -> Dict[str, str]:
    """
    Identifica colunas estruturais com base no schema e em fallbacks.
    """
    def pick(candidates: List[str], default: Optional[str] = None) -> str:
        for c in candidates:
            if c in df_columns:
                return c
        if default is not None:
            return default
        raise ValueError(f"Nenhuma coluna encontrada entre: {candidates}")

    # Tenta ler nomes declarados no schema, se existirem
    target_candidates = [
        schema.get("target_column"),
        schema.get("target_col"),
        "y_nb11",
        "y",
        "target",
    ]
    eligible_candidates = [
        schema.get("endpoint_eligibility_column"),
        schema.get("eligibility_column"),
        "eligible_supervised",
    ]
    scenario_candidates = [
        schema.get("scenario_column"),
        "scenario_label",
        "scenario",
    ]
    order_candidates = [
        schema.get("time_order_column"),
        schema.get("order_column"),
        "bucket_id",
        "row_position",
    ]

    target_candidates = [c for c in target_candidates if isinstance(c, str)]
    eligible_candidates = [c for c in eligible_candidates if isinstance(c, str)]
    scenario_candidates = [c for c in scenario_candidates if isinstance(c, str)]
    order_candidates = [c for c in order_candidates if isinstance(c, str)]

    return {
        "target": pick(target_candidates),
        "eligible": pick(eligible_candidates),
        "scenario": pick(scenario_candidates),
        "order": pick(order_candidates),
    }

def validate_active_features(
    features: List[str],
    df_columns: List[str],
    structural_cols: Dict[str, str],
    feature_sets_obj: Dict[str, Any],
) -> None:
    forbidden_exact = {
        structural_cols["target"],
        structural_cols["eligible"],
        structural_cols["scenario"],
        structural_cols["order"],
        "event_FAIL_count",
        "bucket_start",
        "bucket_start_datetime",
        "bucket_start_dt",
        "bucket_start_us",
        "timestamp",
        "datetime",
        "time",
        "state",
        "episode_state",
        "operational_state",
        "is_critical",
        "is_during",
        "is_before",
        "is_after",
        "is_normal",
        "threshold_value",
        "scenario_id",
        "scenario_key",
        "scenario_name",
        "y_nb13",
        "target_col",
    }
    forbidden_exact.update(
        str(item)
        for item in feature_sets_obj.get("forbidden_predictors", [])
    )

    forbidden_patterns = [
        r"^y$",
        r"^y_",
        r"target",
        r"label",
        r"future",
        r"will_",
        r"ahead",
        r"state",
        r"estado",
        r"phase",
        r"during",
        r"before",
        r"after",
        r"episode",
        r"critical",
        r"normal",
        r"eligible",
        r"horizon",
        r"persistence",
        r"threshold",
        r"limiar",
        r"scenario",
        r"cenario",
        r"bucket_start",
        r"timestamp",
        r"datetime",
        r"row_position",
        r"(^|_)score($|_)",
        r"(^|_)scores($|_)",
        r"(^|_)prob($|_)",
        r"(^|_)proba($|_)",
        r"(^|_)pred($|_)",
        r"(^|_)prediction($|_)",
        r"(^|_)fold($|_)",
        r"(^|_)split($|_)",
    ]
    forbidden_patterns.extend(
        str(item)
        for item in feature_sets_obj.get(
            "forbidden_feature_patterns",
            [],
        )
    )

    missing = [feature for feature in features if feature not in df_columns]
    if missing:
        raise ValueError(
            f"Atributos ausentes no dataframe sequencial: {missing}"
        )

    rejected = []
    for feature in features:
        if feature in forbidden_exact:
            rejected.append((feature, "forbidden_exact"))
        elif any(
            re.search(pattern, feature, flags=re.IGNORECASE)
            for pattern in forbidden_patterns
        ):
            rejected.append((feature, "forbidden_pattern"))

    if rejected:
        raise ValueError(
            f"Atributos proibidos no vetor ativo: {rejected}"
        )

    assert len(features) == 13
    assert len(features) == len(set(features))
    assert "event_FAIL_count" not in features


def validate_provenance_and_rate(df: pd.DataFrame) -> None:
    required = {
        "n_failed",
        "event_FAIL_count",
        "n_events",
        "fail_rate",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(
            f"Colunas obrigatórias de proveniência ausentes: {missing}"
        )

    failed = pd.to_numeric(df["n_failed"], errors="coerce")
    failed_event = pd.to_numeric(
        df["event_FAIL_count"],
        errors="coerce",
    )

    equality = failed.eq(failed_event) | (
        failed.isna() & failed_event.isna()
    )
    if not bool(equality.all()):
        divergent_rows = int((~equality).sum())
        raise ValueError(
            "Divergência entre n_failed e event_FAIL_count: "
            f"{divergent_rows} linhas."
        )

    events = pd.to_numeric(df["n_events"], errors="coerce")
    observed_rate = pd.to_numeric(df["fail_rate"], errors="coerce")
    expected_rate = pd.Series(
        np.divide(
            failed.fillna(0.0).to_numpy(dtype=float),
            events.fillna(0.0).to_numpy(dtype=float),
            out=np.zeros(len(df), dtype=float),
            where=events.fillna(0.0).to_numpy(dtype=float) != 0,
        ),
        index=df.index,
    )

    valid = observed_rate.notna()
    if not np.allclose(
        observed_rate.loc[valid].to_numpy(dtype=float),
        expected_rate.loc[valid].to_numpy(dtype=float),
        rtol=1e-7,
        atol=1e-10,
    ):
        raise ValueError(
            "fail_rate não coincide com n_failed / n_events."
        )


def prepare_model_frame(
    df: pd.DataFrame,
    features: List[str],
) -> pd.DataFrame:
    return (
        df[features]
        .replace([np.inf, -np.inf], np.nan)
        .ffill()
        .fillna(0.0)
        .astype("float32")
    )


def exact_duplicate_pairs(df: pd.DataFrame) -> List[Tuple[str, str]]:
    pairs: List[Tuple[str, str]] = []
    columns = list(df.columns)

    for index, left in enumerate(columns):
        for right in columns[index + 1:]:
            left_series = df[left]
            right_series = df[right]
            equal = (
                left_series.eq(right_series)
                | (left_series.isna() & right_series.isna())
            ).all()
            if bool(equal):
                pairs.append((left, right))

    return pairs


def build_feature_audit(
    scenario_label: str,
    features: List[str],
    model_frame: pd.DataFrame,
) -> pd.DataFrame:
    duplicate_pairs = exact_duplicate_pairs(model_frame)
    if duplicate_pairs:
        raise ValueError(
            "Duplicidade exata não justificada no dataframe modelado: "
            f"{duplicate_pairs}"
        )

    return pd.DataFrame(
        [
            {
                "scenario_label": scenario_label,
                "feature_set": FEATURE_SET_TO_TUNE,
                "n_rows_model_frame": int(len(model_frame)),
                "n_features": int(len(features)),
                "event_FAIL_count_in_predictors": (
                    "event_FAIL_count" in features
                ),
                "n_failed_event_FAIL_count_equality": True,
                "exact_duplicate_pair_count": 0,
                "exact_duplicate_pairs": "",
                "status": "PASS",
            }
        ]
    )


def make_sequences_for_scenario(
    df: pd.DataFrame,
    features: List[str],
    target_col: str,
    eligible_col: str,
    order_col: str,
    sequence_length: int,
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Forma sequências LSTM respeitando a ordem temporal.
    A elegibilidade é aplicada ao endpoint da sequência, não aos pontos internos.
    """
    sdf = df.sort_values(order_col).reset_index(drop=True).copy()

    X_raw = prepare_model_frame(sdf, features)
    y_raw = sdf[target_col].astype(int).values
    eligible = sdf[eligible_col].astype(bool).values

    X_seq = []
    y_seq = []
    meta_rows = []

    for end_idx in range(sequence_length - 1, len(sdf)):
        if not eligible[end_idx]:
            continue

        start_idx = end_idx - sequence_length + 1
        seq = X_raw.iloc[start_idx:end_idx + 1].values

        X_seq.append(seq)
        y_seq.append(y_raw[end_idx])
        meta_rows.append({
            "row_position": int(end_idx),
            "bucket_id": int(sdf.loc[end_idx, order_col]) if pd.notna(sdf.loc[end_idx, order_col]) else int(end_idx),
            "y_true": int(y_raw[end_idx]),
        })

    X_seq = np.asarray(X_seq, dtype="float32")
    y_seq = np.asarray(y_seq, dtype=int)
    meta = pd.DataFrame(meta_rows)

    if X_seq.ndim != 3:
        raise ValueError(
            f"Tensor LSTM inválido: ndim={X_seq.ndim}; esperado=3."
        )
    if X_seq.shape[-1] != len(features):
        raise ValueError(
            "Dimensão final do tensor difere do vetor de atributos: "
            f"{X_seq.shape[-1]} != {len(features)}."
        )
    assert X_seq.shape[-1] == 13

    return X_seq, y_seq, meta

def temporal_train_val_split(X_train: np.ndarray, y_train: np.ndarray, val_fraction: float):
    n = len(y_train)
    n_val = max(1, int(round(n * val_fraction)))
    n_train_inner = n - n_val

    # Garante pelo menos algumas amostras para treino
    if n_train_inner < 50:
        n_train_inner = max(1, n - 1)
        n_val = n - n_train_inner

    return (
        X_train[:n_train_inner],
        y_train[:n_train_inner],
        X_train[n_train_inner:],
        y_train[n_train_inner:],
    )

def scale_sequences(X_train: np.ndarray, X_val: np.ndarray, X_test: np.ndarray):
    n_train, seq_len, n_feat = X_train.shape

    scaler = StandardScaler()
    X_train_2d = X_train.reshape(-1, n_feat)
    scaler.fit(X_train_2d)

    def transform(X):
        shape = X.shape
        return scaler.transform(X.reshape(-1, n_feat)).reshape(shape).astype("float32")

    return transform(X_train), transform(X_val), transform(X_test), scaler

def compute_class_weight_dict(y: np.ndarray) -> Optional[Dict[int, float]]:
    classes, counts = np.unique(y, return_counts=True)
    if len(classes) < 2:
        return None
    n = len(y)
    weights = {int(cls): float(n / (len(classes) * cnt)) for cls, cnt in zip(classes, counts)}
    return weights

def build_lstm_model(sequence_length: int, n_features: int, units: int, dropout: float, learning_rate: float):
    model = models.Sequential([
        layers.Input(shape=(sequence_length, n_features)),
        layers.LSTM(units),
        layers.Dropout(dropout),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[],
    )
    return model

def calculate_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> Dict[str, Any]:
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "n": int(len(y_true)),
        "n_positive": int(np.sum(y_true)),
        "positive_rate": float(np.mean(y_true)) if len(y_true) else None,
        "accuracy": safe_float(accuracy_score(y_true, y_pred)),
        "precision": safe_float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": safe_float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": safe_float(f1_score(y_true, y_pred, zero_division=0)),
        "brier": safe_float(brier_score_loss(y_true, y_score)),
    }

    try:
        out["roc_auc"] = safe_float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else None
    except Exception:
        out["roc_auc"] = None

    try:
        out["average_precision"] = safe_float(average_precision_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else None
    except Exception:
        out["average_precision"] = None

    return out

def train_and_evaluate_once(
    X_train_full: np.ndarray,
    y_train_full: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    sequence_length: int,
    n_features: int,
    units: int,
    dropout: float,
    learning_rate: float,
    seed: int,
) -> Dict[str, Any]:
    set_global_seed(seed)
    tf.keras.backend.clear_session()

    X_train_inner, y_train_inner, X_val, y_val = temporal_train_val_split(
        X_train_full, y_train_full, VALIDATION_FRACTION_FROM_TRAIN
    )

    X_train_scaled, X_val_scaled, X_test_scaled, _ = scale_sequences(X_train_inner, X_val, X_test)

    model = build_lstm_model(sequence_length, n_features, units, dropout, learning_rate)

    cb = [
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            min_delta=1e-5,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            patience=REDUCE_LR_PATIENCE,
            factor=0.5,
            min_lr=1e-6,
        ),
    ]

    class_weight = compute_class_weight_dict(y_train_inner)

    t0 = time.time()
    hist = model.fit(
        X_train_scaled,
        y_train_inner,
        validation_data=(X_val_scaled, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=KERAS_VERBOSE,
        callbacks=cb,
        class_weight=class_weight,
        shuffle=False,
    )
    fit_elapsed = time.time() - t0

    y_score = model.predict(X_test_scaled, verbose=0).reshape(-1)
    metrics = calculate_metrics(y_test, y_score, CLASSIFICATION_THRESHOLD)

    metrics.update({
        "epochs_ran": int(len(hist.history.get("loss", []))),
        "best_val_loss": float(np.min(hist.history.get("val_loss", [np.nan]))),
        "fit_elapsed_seconds": float(fit_elapsed),
        "total_elapsed_seconds": float(time.time() - t0),
    })

    return metrics | {"y_score": y_score}

def aggregate_results(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    metric_cols = [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision",
        "brier",
        "n",
        "n_positive",
        "positive_rate",
        "epochs_ran",
        "best_val_loss",
        "fit_elapsed_seconds",
        "total_elapsed_seconds",
    ]

    agg_map = {}
    for c in metric_cols:
        if c in df.columns:
            agg_map[c] = ["mean", "std"]

    out = df.groupby(group_cols, dropna=False).agg(agg_map)
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    out = out.reset_index()

    # Ordenação para seleção
    sort_cols = []
    for c in ["f1_mean", "recall_mean", "roc_auc_mean", "average_precision_mean"]:
        if c in out.columns:
            sort_cols.append(c)

    if sort_cols:
        out = out.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)

    return out

def run_tscv_for_config(
    X: np.ndarray,
    y: np.ndarray,
    config: Dict[str, Any],
    seeds: List[int],
    stage: str,
) -> pd.DataFrame:
    rows = []
    tscv = TimeSeriesSplit(n_splits=TSCV_SPLITS)

    for seed in seeds:
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
            y_train = y[train_idx]
            y_test = y[test_idx]

            if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                row = {
                    "stage": stage,
                    "seed": seed,
                    "fold": fold,
                    "valid_fold": False,
                    "skip_reason": "single_class_train_or_test",
                    **config,
                }
                rows.append(row)
                continue

            result = train_and_evaluate_once(
                X_train_full=X[train_idx],
                y_train_full=y_train,
                X_test=X[test_idx],
                y_test=y_test,
                sequence_length=config["sequence_length"],
                n_features=X.shape[-1],
                units=config["units"],
                dropout=config["dropout"],
                learning_rate=config["learning_rate"],
                seed=seed,
            )

            y_score = result.pop("y_score")
            row = {
                "stage": stage,
                "seed": seed,
                "fold": fold,
                "valid_fold": True,
                "skip_reason": "",
                **config,
                **result,
            }
            rows.append(row)

    return pd.DataFrame(rows)

def run_fixed_for_config(
    X: np.ndarray,
    y: np.ndarray,
    meta: pd.DataFrame,
    config: Dict[str, Any],
    seeds: List[int],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    n = len(y)
    split_idx = int(math.floor(n * FIXED_TRAIN_FRACTION))

    X_train = X[:split_idx]
    y_train = y[:split_idx]
    X_test = X[split_idx:]
    y_test = y[split_idx:]
    meta_test = meta.iloc[split_idx:].reset_index(drop=True).copy()

    rows = []
    score_frames = []

    for seed in seeds:
        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            rows.append({
                "seed": seed,
                "valid_split": False,
                "skip_reason": "single_class_train_or_test",
                **config,
            })
            continue

        result = train_and_evaluate_once(
            X_train_full=X_train,
            y_train_full=y_train,
            X_test=X_test,
            y_test=y_test,
            sequence_length=config["sequence_length"],
            n_features=X.shape[-1],
            units=config["units"],
            dropout=config["dropout"],
            learning_rate=config["learning_rate"],
            seed=seed,
        )

        y_score = result.pop("y_score")
        rows.append({
            "seed": seed,
            "valid_split": True,
            "skip_reason": "",
            **config,
            **result,
        })

        sf = meta_test.copy()
        sf["seed"] = seed
        sf["score_lstm_tuned"] = y_score
        sf["y_pred_lstm_tuned"] = (y_score >= CLASSIFICATION_THRESHOLD).astype(int)
        for k, v in config.items():
            sf[k] = v
        score_frames.append(sf)

    fixed_results = pd.DataFrame(rows)
    scores = pd.concat(score_frames, ignore_index=True) if score_frames else pd.DataFrame()
    return fixed_results, scores

# ------------------------------------------------------------
# 4. Leitura e preflight integral
# ------------------------------------------------------------

required_inputs = [
    SEQUENCE_INPUT_PATH,
    FEATURE_SETS_PATH,
    SCHEMA_PATH,
    NB11_SUMMARY_PATH,
    WINNER_MODEL_PATH,
    NB13_SUMMARY_PATH,
]

for path in required_inputs:
    if not path.exists():
        raise FileNotFoundError(
            f"Entrada obrigatória não encontrada: {path}"
        )

print("[INFO] Lendo artefatos...")
df_seq = pd.read_parquet(SEQUENCE_INPUT_PATH)
feature_sets_obj = load_json(FEATURE_SETS_PATH)
schema_obj = load_json(SCHEMA_PATH)
nb11_summary = load_json(NB11_SUMMARY_PATH)
winner_model = load_json(WINNER_MODEL_PATH)
nb13_summary = load_json(NB13_SUMMARY_PATH)

struct_cols = detect_schema_columns(
    schema_obj,
    list(df_seq.columns),
)
print("[INFO] Colunas estruturais:", struct_cols)

main_scenario = (
    winner_model.get("scenario_label")
    or winner_model.get("scenario")
)
if not main_scenario:
    raise ValueError(
        "O cenário principal não foi encontrado em 11_winner_model.json."
    )
if main_scenario != MAIN_SCENARIO_DEFAULT:
    raise ValueError(
        "Cenário principal inesperado: "
        f"{main_scenario} != {MAIN_SCENARIO_DEFAULT}."
    )

features = extract_feature_set(
    feature_sets_obj,
    main_scenario,
    FEATURE_SET_TO_TUNE,
)
validate_feature_contract(
    feature_sets_obj=feature_sets_obj,
    schema_obj=schema_obj,
    scenario_label=main_scenario,
    active_features=features,
)
validate_active_features(
    features=features,
    df_columns=list(df_seq.columns),
    structural_cols=struct_cols,
    feature_sets_obj=feature_sets_obj,
)
validate_provenance_and_rate(df_seq)

print("[INFO] Cenário principal:", main_scenario)
print(
    f"[INFO] Conjunto avaliado: {FEATURE_SET_TO_TUNE} "
    f"({len(features)} atributos)"
)
print(features)

# ------------------------------------------------------------
# 5. Preparação e auditoria da base sequencial
# ------------------------------------------------------------

scenario_col = struct_cols["scenario"]
target_col = struct_cols["target"]
eligible_col = struct_cols["eligible"]
order_col = struct_cols["order"]

df_main = (
    df_seq[df_seq[scenario_col] == main_scenario]
    .sort_values(order_col)
    .reset_index(drop=True)
    .copy()
)
if df_main.empty:
    raise ValueError(
        f"Nenhuma linha encontrada para scenario_label={main_scenario}"
    )

model_frame = prepare_model_frame(df_main, features)
feature_audit = build_feature_audit(
    scenario_label=main_scenario,
    features=features,
    model_frame=model_frame,
)

diagnostics_by_L: Dict[str, Dict[str, Any]] = {}
sequence_data: Dict[int, Tuple[np.ndarray, np.ndarray, pd.DataFrame]] = {}

for sequence_length in SEQUENCE_LENGTH_GRID:
    X, y, meta = make_sequences_for_scenario(
        df=df_main,
        features=features,
        target_col=target_col,
        eligible_col=eligible_col,
        order_col=order_col,
        sequence_length=sequence_length,
    )
    if len(y) == 0:
        raise ValueError(
            "Nenhuma sequência formada para "
            f"sequence_length={sequence_length}"
        )

    diagnostics_by_L[str(sequence_length)] = {
        "sequence_length": int(sequence_length),
        "n_sequences": int(len(y)),
        "n_positive": int(np.sum(y)),
        "positive_rate": float(np.mean(y)),
        "n_features": int(X.shape[-1]),
        "tensor_shape": list(X.shape),
    }
    sequence_data[sequence_length] = (X, y, meta)

tensor_dimensions = sorted(
    {
        int(value["n_features"])
        for value in diagnostics_by_L.values()
    }
)
if tensor_dimensions != [13]:
    raise ValueError(
        "Dimensões de atributos inesperadas nos tensores: "
        f"{tensor_dimensions}"
    )

feature_audit["sequence_lengths_checked"] = "|".join(
    str(value) for value in SEQUENCE_LENGTH_GRID
)
feature_audit["tensor_feature_dimensions"] = "|".join(
    str(diagnostics_by_L[str(value)]["n_features"])
    for value in SEQUENCE_LENGTH_GRID
)

print("[INFO] Diagnóstico por sequence_length:")
print(
    json.dumps(
        diagnostics_by_L,
        ensure_ascii=False,
        indent=2,
    )
)

# ------------------------------------------------------------
# 6. Referências autoritativas do NB11 e do NB13
# ------------------------------------------------------------

def extract_nb11_reference(
    nb11_summary_obj: Dict[str, Any],
    winner_model_obj: Dict[str, Any],
) -> Dict[str, Any]:
    required = [
        "scenario_label",
        "model",
        "feature_set",
        "f1_tscv_mean",
    ]
    missing = [
        key
        for key in required
        if winner_model_obj.get(key) is None
    ]
    if missing:
        raise ValueError(
            "Campos ausentes em 11_winner_model.json: "
            f"{missing}"
        )

    features_ref = winner_model_obj.get("features", [])
    if features_ref and list(features_ref) != TEMPORAL_CORE:
        raise ValueError(
            "A lista de atributos do vencedor do NB11 "
            "difere de temporal_core."
        )

    if winner_model_obj["scenario_label"] != main_scenario:
        raise ValueError(
            "O cenário do vencedor do NB11 difere do cenário ativo."
        )
    if winner_model_obj["feature_set"] != FEATURE_SET_TO_TUNE:
        raise ValueError(
            "O conjunto do vencedor do NB11 não é temporal_core."
        )
    if int(winner_model_obj.get("n_features", 13)) != 13:
        raise ValueError(
            "O vencedor do NB11 não registra 13 atributos."
        )

    return {
        "scenario_label": winner_model_obj["scenario_label"],
        "model": winner_model_obj["model"],
        "feature_set": winner_model_obj["feature_set"],
        "f1_tscv_mean": float(
            winner_model_obj["f1_tscv_mean"]
        ),
        "f1_tscv_std": safe_float(
            winner_model_obj.get("f1_tscv_std")
        ),
        "n_features": 13,
    }


def extract_nb13_reference(
    nb13_summary_obj: Dict[str, Any],
) -> Dict[str, Any]:
    best = nb13_summary_obj.get("best_lstm_tscv")
    if not isinstance(best, dict) or not best:
        raise ValueError(
            "13_lstm_summary.json não contém best_lstm_tscv."
        )

    f1 = nb13_summary_obj.get(
        "best_lstm_f1_tscv_mean"
    )
    if f1 is None:
        f1 = best.get("f1_mean")
    if f1 is None:
        raise ValueError(
            "F1 TSCV da melhor LSTM não encontrado no NB13."
        )

    contract = nb13_summary_obj.get("feature_contract", {})
    if contract:
        if int(
            contract.get("raw_core_n_features", 6)
        ) != 6:
            raise ValueError(
                "Contrato raw_core do NB13 não registra 6 atributos."
            )
        if int(
            contract.get("temporal_core_n_features", 13)
        ) != 13:
            raise ValueError(
                "Contrato temporal_core do NB13 não registra 13 atributos."
            )
        if contract.get("raw_core") != RAW_CORE:
            raise ValueError(
                "raw_core do NB13 difere do contrato normativo."
            )
        if contract.get("temporal_core") != TEMPORAL_CORE:
            raise ValueError(
                "temporal_core do NB13 difere do contrato normativo."
            )

    if best.get("scenario_label") != main_scenario:
        raise ValueError(
            "A melhor LSTM do NB13 não pertence ao cenário principal."
        )
    if best.get("feature_set") != FEATURE_SET_TO_TUNE:
        raise ValueError(
            "A melhor LSTM do NB13 não usa temporal_core."
        )

    return {
        "available": True,
        "best_lstm_tscv": best,
        "best_lstm_f1_tscv_mean": float(f1),
        "feed_nb14_decision": nb13_summary_obj.get(
            "feed_nb14_decision"
        ),
        "n_features": 13,
    }


nb11_reference = extract_nb11_reference(
    nb11_summary,
    winner_model,
)
nb13_reference = extract_nb13_reference(nb13_summary)

print("[INFO] Referência NB11:", nb11_reference)
print("[INFO] Referência NB13:", nb13_reference)

# ------------------------------------------------------------
# 7. Preservação e limpeza dos outputs ativos do NB13a
# ------------------------------------------------------------

backup_dir, removed_previous_outputs = (
    backup_and_clear_existing_13a_outputs()
)
feature_audit.to_csv(OUT_FEATURE_AUDIT, index=False)

if backup_dir:
    print(
        "[INFO] Outputs 13a anteriores preservados em:",
        backup_dir,
    )
if removed_previous_outputs:
    print(
        "[INFO] Outputs ativos anteriores removidos:",
        len(removed_previous_outputs),
    )
print("[OK] Preflight concluído antes do treinamento.")

# ------------------------------------------------------------
# 8. Stage 1 — triagem do grid completo
# ------------------------------------------------------------

grid_configs = [
    {
        "scenario_label": main_scenario,
        "feature_set": FEATURE_SET_TO_TUNE,
        "sequence_length": L,
        "units": u,
        "dropout": d,
        "learning_rate": lr,
    }
    for L, u, d, lr in product(SEQUENCE_LENGTH_GRID, LSTM_UNITS_GRID, DROPOUT_GRID, LEARNING_RATE_GRID)
]

print(f"[INFO] Stage 1: {len(grid_configs)} configurações × {len(STAGE1_SEEDS)} seed(s) × {TSCV_SPLITS} folds")

stage1_frames = []
stage1_start = time.time()

for idx, cfg in enumerate(grid_configs, start=1):
    print(f"[STAGE1] {idx:03d}/{len(grid_configs)} — L={cfg['sequence_length']} units={cfg['units']} dropout={cfg['dropout']} lr={cfg['learning_rate']}")
    X, y, _meta = sequence_data[cfg["sequence_length"]]
    df_res = run_tscv_for_config(X, y, cfg, STAGE1_SEEDS, stage="stage1_screening")
    stage1_frames.append(df_res)

stage1_results = pd.concat(stage1_frames, ignore_index=True)
stage1_results.to_csv(OUT_STAGE1_RESULTS, index=False)

valid_stage1 = stage1_results[
    stage1_results["valid_fold"] == True
].copy()
if valid_stage1.empty:
    raise RuntimeError(
        "Stage 1 não produziu folds válidos."
    )

stage1_agg = aggregate_results(
    valid_stage1,
    group_cols=["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"],
)
stage1_agg.to_csv(OUT_STAGE1_AGG, index=False)

print(f"[INFO] Stage 1 concluído em {(time.time() - stage1_start)/60:.2f} min")
print("[INFO] Top configurações Stage 1:")
print(stage1_agg.head(TOP_K_FOR_CONFIRMATION)[[
    "sequence_length", "units", "dropout", "learning_rate", "f1_mean", "recall_mean", "precision_mean", "roc_auc_mean"
]])

# ------------------------------------------------------------
# 9. Stage 2 — confirmação das melhores configurações com múltiplas sementes
# ------------------------------------------------------------

top_configs_records = stage1_agg.head(TOP_K_FOR_CONFIRMATION)[
    ["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"]
].to_dict(orient="records")

print(f"[INFO] Stage 2: top {len(top_configs_records)} configurações × {len(STAGE2_SEEDS)} seeds × {TSCV_SPLITS} folds")

stage2_frames = []
stage2_start = time.time()

for idx, cfg in enumerate(top_configs_records, start=1):
    print(f"[STAGE2] {idx:03d}/{len(top_configs_records)} — L={cfg['sequence_length']} units={cfg['units']} dropout={cfg['dropout']} lr={cfg['learning_rate']}")
    X, y, _meta = sequence_data[int(cfg["sequence_length"])]
    df_res = run_tscv_for_config(X, y, cfg, STAGE2_SEEDS, stage="stage2_confirmation")
    stage2_frames.append(df_res)

stage2_results = pd.concat(stage2_frames, ignore_index=True)
stage2_results.to_csv(OUT_STAGE2_RESULTS, index=False)

valid_stage2 = stage2_results[
    stage2_results["valid_fold"] == True
].copy()
if valid_stage2.empty:
    raise RuntimeError(
        "Stage 2 não produziu folds válidos."
    )

stage2_agg = aggregate_results(
    valid_stage2,
    group_cols=["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"],
)
stage2_agg.to_csv(OUT_STAGE2_AGG, index=False)

print(f"[INFO] Stage 2 concluído em {(time.time() - stage2_start)/60:.2f} min")
print("[INFO] Top configurações Stage 2:")
print(stage2_agg.head(10)[[
    "sequence_length", "units", "dropout", "learning_rate", "f1_mean", "f1_std", "recall_mean", "precision_mean", "roc_auc_mean"
]])

# ------------------------------------------------------------
# 10. Seleção final e avaliação complementar no corte fixo
# ------------------------------------------------------------

if stage2_agg.empty:
    raise RuntimeError("Stage 2 não produziu resultados válidos.")

best_row = stage2_agg.iloc[0].to_dict()

best_config = {
    "scenario_label": main_scenario,
    "feature_set": FEATURE_SET_TO_TUNE,
    "sequence_length": int(best_row["sequence_length"]),
    "units": int(best_row["units"]),
    "dropout": float(best_row["dropout"]),
    "learning_rate": float(best_row["learning_rate"]),
}

X_best, y_best, meta_best = sequence_data[best_config["sequence_length"]]
fixed_results, scores = run_fixed_for_config(X_best, y_best, meta_best, best_config, STAGE2_SEEDS)
fixed_results.to_csv(OUT_FIXED_RESULTS, index=False)

valid_fixed = fixed_results[fixed_results["valid_split"] == True].copy()
fixed_agg = aggregate_results(
    valid_fixed,
    group_cols=["scenario_label", "feature_set", "sequence_length", "units", "dropout", "learning_rate"],
)
fixed_agg.to_csv(OUT_FIXED_AGG, index=False)

if not scores.empty:
    # Cria escore médio por endpoint para facilitar análise visual futura
    score_mean = (
        scores.groupby(["row_position", "bucket_id", "y_true"], as_index=False)
        .agg(
            score_lstm_tuned_mean=("score_lstm_tuned", "mean"),
            score_lstm_tuned_std=("score_lstm_tuned", "std"),
        )
    )
    for k, v in best_config.items():
        score_mean[k] = v
    score_mean["classification_threshold"] = CLASSIFICATION_THRESHOLD
    score_mean["y_pred_lstm_tuned_mean"] = (score_mean["score_lstm_tuned_mean"] >= CLASSIFICATION_THRESHOLD).astype(int)
    score_mean.to_parquet(OUT_SCORES, index=False)

# ------------------------------------------------------------
# 11. Decisão e síntese
# ------------------------------------------------------------

best_stage2_metrics = {
    k: safe_float(best_row.get(k))
    for k in [
        "accuracy_mean", "accuracy_std",
        "precision_mean", "precision_std",
        "recall_mean", "recall_std",
        "f1_mean", "f1_std",
        "roc_auc_mean", "roc_auc_std",
        "average_precision_mean", "average_precision_std",
        "brier_mean", "brier_std",
        "epochs_ran_mean", "epochs_ran_std",
        "best_val_loss_mean", "best_val_loss_std",
        "fit_elapsed_seconds_mean", "total_elapsed_seconds_mean",
    ]
}

best_f1 = best_stage2_metrics.get("f1_mean")
nb11_f1 = nb11_reference["f1_tscv_mean"]
nb13_f1 = nb13_reference["best_lstm_f1_tscv_mean"]

delta_vs_nb11 = best_f1 - nb11_f1 if best_f1 is not None else None
delta_vs_nb13 = best_f1 - nb13_f1 if best_f1 is not None else None

if delta_vs_nb11 is not None and delta_vs_nb11 >= MIN_DELTA_F1_TO_FEED_NB14:
    feed_nb14_decision = "candidate_lstm_tuned_scores_for_nb14"
else:
    feed_nb14_decision = "keep_nb11_scores_for_nb14"

if delta_vs_nb13 is not None and delta_vs_nb13 >= MIN_DELTA_F1_TO_CONSIDER_TUNING_GAIN:
    tuning_interpretation = "compact_tuning_improved_lstm_materially"
elif delta_vs_nb13 is not None and delta_vs_nb13 > 0:
    tuning_interpretation = "compact_tuning_improved_lstm_slightly"
else:
    tuning_interpretation = "compact_tuning_did_not_improve_lstm"

best_config_payload = {
    "created_at": now_str(),
    "selection_protocol": "stage2_confirmation_timeseries_split",
    "selection_metric": "f1_mean",
    "best_config": best_config,
    "best_stage2_metrics": best_stage2_metrics,
    "nb11_reference": nb11_reference,
    "nb13_reference": nb13_reference,
    "delta_f1_tuned_lstm_minus_nb11": delta_vs_nb11,
    "delta_f1_tuned_lstm_minus_nb13": delta_vs_nb13,
    "min_delta_f1_to_feed_nb14": MIN_DELTA_F1_TO_FEED_NB14,
    "feed_nb14_decision": feed_nb14_decision,
    "tuning_interpretation": tuning_interpretation,
}
save_json(best_config_payload, OUT_BEST_CONFIG)

nominal_evaluations = {
    "stage1": int(
        len(grid_configs)
        * len(STAGE1_SEEDS)
        * TSCV_SPLITS
    ),
    "stage2": int(
        len(top_configs_records)
        * len(STAGE2_SEEDS)
        * TSCV_SPLITS
    ),
    "fixed_complementary": int(len(STAGE2_SEEDS)),
}
nominal_evaluations["total"] = int(
    sum(nominal_evaluations.values())
)

actual_evaluations = {
    "stage1_total": int(len(stage1_results)),
    "stage1_valid": int(len(valid_stage1)),
    "stage1_skipped": int(
        len(stage1_results) - len(valid_stage1)
    ),
    "stage2_total": int(len(stage2_results)),
    "stage2_valid": int(len(valid_stage2)),
    "stage2_skipped": int(
        len(stage2_results) - len(valid_stage2)
    ),
    "fixed_total": int(len(fixed_results)),
    "fixed_valid": int(len(valid_fixed)),
    "fixed_skipped": int(
        len(fixed_results) - len(valid_fixed)
    ),
}
actual_evaluations["total"] = int(
    actual_evaluations["stage1_total"]
    + actual_evaluations["stage2_total"]
    + actual_evaluations["fixed_total"]
)
actual_evaluations["valid_total"] = int(
    actual_evaluations["stage1_valid"]
    + actual_evaluations["stage2_valid"]
    + actual_evaluations["fixed_valid"]
)
actual_evaluations["skipped_total"] = int(
    actual_evaluations["stage1_skipped"]
    + actual_evaluations["stage2_skipped"]
    + actual_evaluations["fixed_skipped"]
)

summary = {
    "notebook": "NB13a",
    "title": "Tuning compacto da LSTM",
    "executed_at": now_str(),
    "objective": (
        "Avaliar se um tuning compacto de hiperparâmetros da LSTM altera a conclusão do NB13 "
        "sobre a manutenção dos escores do modelo tabular vencedor do NB11 no NB14."
    ),
    "intermediate_notebook_rule": {
        "can_read_previous_artifacts": True,
        "must_not_overwrite_parent_outputs": True,
        "output_prefix": "13a_",
    },
    "config": {
        "run_mode": RUN_MODE,
        "main_scenario": main_scenario,
        "feature_set": FEATURE_SET_TO_TUNE,
        "sequence_length_grid": SEQUENCE_LENGTH_GRID,
        "lstm_units_grid": LSTM_UNITS_GRID,
        "dropout_grid": DROPOUT_GRID,
        "learning_rate_grid": LEARNING_RATE_GRID,
        "stage1_seeds": STAGE1_SEEDS,
        "stage2_seeds": STAGE2_SEEDS,
        "top_k_for_confirmation": TOP_K_FOR_CONFIRMATION,
        "tscv_splits": TSCV_SPLITS,
        "max_epochs": MAX_EPOCHS,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE,
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "fixed_train_fraction": FIXED_TRAIN_FRACTION,
        "validation_fraction_from_train": VALIDATION_FRACTION_FROM_TRAIN,
        "min_delta_f1_to_feed_nb14": MIN_DELTA_F1_TO_FEED_NB14,
    },
    "input_artifacts": {
        "sequence_input_features": str(SEQUENCE_INPUT_PATH),
        "model_input_features": str(MODEL_INPUT_PATH),
        "feature_sets": str(FEATURE_SETS_PATH),
        "schema": str(SCHEMA_PATH),
        "nb11_summary": str(NB11_SUMMARY_PATH),
        "winner_model": str(WINNER_MODEL_PATH),
        "nb13_summary": str(NB13_SUMMARY_PATH),
    },
    "observed_shapes": {
        "sequence_input_shape": list(df_seq.shape),
        "main_scenario_rows": int(len(df_main)),
    },
    "structural_columns": struct_cols,
    "feature_plan": {
        "feature_set": FEATURE_SET_TO_TUNE,
        "n_features": len(features),
        "features": features,
    },
    "feature_contract": {
        "raw_core": RAW_CORE,
        "raw_core_n_features": len(RAW_CORE),
        "temporal_core": TEMPORAL_CORE,
        "temporal_core_n_features": len(TEMPORAL_CORE),
        "event_FAIL_count_preserved_for_provenance": True,
        "event_FAIL_count_excluded_from_predictors": True,
        "n_failed_event_FAIL_count_equality_verified": True,
        "exact_duplicate_check": "PASS before training",
        "tensor_dimension_check": "PASS before training",
    },
    "diagnostics_by_sequence_length": diagnostics_by_L,
    "nominal_evaluation_plan": nominal_evaluations,
    "actual_evaluations": actual_evaluations,
    "best_config": best_config,
    "best_stage2_metrics": best_stage2_metrics,
    "nb11_reference": nb11_reference,
    "nb13_reference": nb13_reference,
    "delta_f1_tuned_lstm_minus_nb11": delta_vs_nb11,
    "delta_f1_tuned_lstm_minus_nb13": delta_vs_nb13,
    "feed_nb14_decision": feed_nb14_decision,
    "tuning_interpretation": tuning_interpretation,
    "outputs": {
        "feature_duplicate_audit": str(OUT_FEATURE_AUDIT),
        "stage1_results": str(OUT_STAGE1_RESULTS),
        "stage1_agg": str(OUT_STAGE1_AGG),
        "stage2_results": str(OUT_STAGE2_RESULTS),
        "stage2_agg": str(OUT_STAGE2_AGG),
        "fixed_results": str(OUT_FIXED_RESULTS),
        "fixed_agg": str(OUT_FIXED_AGG),
        "scores": str(OUT_SCORES),
        "best_config": str(OUT_BEST_CONFIG),
        "summary": str(OUT_SUMMARY),
        "conclusion": str(OUT_CONCLUSION),
    },
    "backup_previous_13a_outputs": (
        str(backup_dir) if backup_dir else None
    ),
    "removed_previous_outputs_after_backup": (
        removed_previous_outputs
    ),
    "figure_policy": {
        "own_figure_required": False,
        "figures_directory": str(FIGURES_PATH),
        "expected_state": "intentional_empty",
    },
    "methodological_notes": [
        "O tuning é restrito ao cenário principal e ao feature set temporal_core, pois este é o ponto de comparação mais forte com o vencedor do NB11.",
        "A seleção usa TimeSeriesSplit como protocolo principal.",
        "O corte fixo 80/20 é complementar e aplicado apenas à configuração final selecionada.",
        "O escalonamento é ajustado apenas no trecho de treino de cada fold.",
        "A validação interna para early stopping usa a parte final do treino, preservando a ordem temporal.",
        "A coluna eligible_supervised é usada apenas para selecionar endpoints elegíveis, não como feature.",
        "Nenhum arquivo 13_ ou de notebooks anteriores é sobrescrito.",
    ],
}
save_json(summary, OUT_SUMMARY)

# Conclusão textual
def fmt(v, nd=4):
    if v is None:
        return "n/d"
    try:
        return f"{float(v):.{nd}f}"
    except Exception:
        return str(v)

conclusion_lines = [
    "Conclusão do NB13a:",
    "",
    (
        "O NB13a executou tuning compacto da LSTM como "
        "evidência complementar ao NB13, sem sobrescrever "
        "artefatos do notebook pai."
    ),
    "",
    (
        f"O escopo foi restrito ao cenário {main_scenario} "
        f"e ao conjunto {FEATURE_SET_TO_TUNE}, com "
        f"{len(features)} atributos."
    ),
    (
        "A triagem produziu "
        f"{actual_evaluations['stage1_total']} avaliações, "
        f"das quais {actual_evaluations['stage1_valid']} "
        "foram válidas."
    ),
    (
        "A confirmação produziu "
        f"{actual_evaluations['stage2_total']} avaliações, "
        f"das quais {actual_evaluations['stage2_valid']} "
        "foram válidas."
    ),
    (
        "O corte fixo complementar produziu "
        f"{actual_evaluations['fixed_total']} avaliações, "
        f"das quais {actual_evaluations['fixed_valid']} "
        "foram válidas."
    ),
    "",
    "Melhor configuração confirmada:",
    f"- sequence_length = {best_config['sequence_length']}",
    f"- units = {best_config['units']}",
    f"- dropout = {best_config['dropout']}",
    f"- learning_rate = {best_config['learning_rate']}",
    "",
    "Desempenho da melhor LSTM ajustada no TimeSeriesSplit:",
    f"- F1 médio = {fmt(best_stage2_metrics.get('f1_mean'))}",
    (
        "- desvio-padrão do F1 = "
        f"{fmt(best_stage2_metrics.get('f1_std'))}"
    ),
    (
        "- recall médio = "
        f"{fmt(best_stage2_metrics.get('recall_mean'))}"
    ),
    (
        "- precisão média = "
        f"{fmt(best_stage2_metrics.get('precision_mean'))}"
    ),
    (
        "- ROC-AUC médio = "
        f"{fmt(best_stage2_metrics.get('roc_auc_mean'))}"
    ),
    (
        "- Brier médio = "
        f"{fmt(best_stage2_metrics.get('brier_mean'))}"
    ),
    "",
    f"Referência NB11: F1 médio TSCV = {fmt(nb11_f1)}.",
    (
        "Referência NB13: melhor F1 médio TSCV da LSTM = "
        f"{fmt(nb13_f1)}."
    ),
    (
        "Delta da LSTM ajustada contra NB11 = "
        f"{fmt(delta_vs_nb11)}."
    ),
    (
        "Delta da LSTM ajustada contra NB13 = "
        f"{fmt(delta_vs_nb13)}."
    ),
    "",
    f"Decisão para NB14: {feed_nb14_decision}.",
    "",
]

if feed_nb14_decision == "candidate_lstm_tuned_scores_for_nb14":
    conclusion_lines.append(
        "A LSTM ajustada superou o modelo tabular vencedor do NB11 pela margem mínima definida. "
        "Recomenda-se avaliar, no NB14, se seus escores devem ser considerados como alternativa decisória."
    )
else:
    conclusion_lines.append(
        "A LSTM ajustada não superou o modelo tabular vencedor do NB11 pela margem mínima definida. "
        "Assim, os escores do NB11 permanecem como entrada preferencial do NB14."
    )

conclusion_lines.extend([
    "",
    "O resultado deve ser interpretado como tuning compacto e controlado, não como busca neural exaustiva.",
    "Ele fortalece a discussão metodológica da dissertação ao mostrar se a conclusão do NB13 se mantém ou não após ajuste moderado de hiperparâmetros.",
])

with open(OUT_CONCLUSION, "w", encoding="utf-8") as f:
    f.write("\n".join(conclusion_lines))

print("\n".join(conclusion_lines))

# ------------------------------------------------------------
# 12. Conferência final de outputs
# ------------------------------------------------------------

declared_outputs = [
    OUT_FEATURE_AUDIT,
    OUT_STAGE1_RESULTS,
    OUT_STAGE1_AGG,
    OUT_STAGE2_RESULTS,
    OUT_STAGE2_AGG,
    OUT_FIXED_RESULTS,
    OUT_FIXED_AGG,
    OUT_BEST_CONFIG,
    OUT_SUMMARY,
    OUT_CONCLUSION,
]

if len(valid_fixed) > 0:
    declared_outputs.append(OUT_SCORES)

missing_outputs = [
    str(path)
    for path in declared_outputs
    if not path.exists()
]
figures_directory_empty = (
    FIGURES_PATH.exists()
    and not any(FIGURES_PATH.iterdir())
)

summary["all_declared_outputs_exist"] = (
    len(missing_outputs) == 0
)
summary["missing_outputs"] = missing_outputs
summary["figures_directory_empty_intentional"] = (
    figures_directory_empty
)
save_json(summary, OUT_SUMMARY)

if missing_outputs:
    raise RuntimeError(
        f"Outputs declarados ausentes: {missing_outputs}"
    )

if not figures_directory_empty:
    raise RuntimeError(
        "figures_nb13a_lstm_tuning deveria permanecer vazio."
    )

print("[OK] Todos os outputs declarados foram gerados.")
print(
    "[OK] figures_nb13a_lstm_tuning permanece vazio "
    "por decisão editorial."
)



Mounted at /content/drive
TensorFlow: 2.20.0
[INFO] Lendo artefatos...
[INFO] Colunas estruturais: {'target': 'y_nb11', 'eligible': 'eligible_supervised', 'scenario': 'scenario_label', 'order': 'bucket_id'}
[INFO] Cenário principal: W5_K24_H12_P1_TRAIN_M2S
[INFO] Conjunto avaliado: temporal_core (13 atributos)
['fail_rate', 'n_events', 'n_failed', 'n_machines', 'n_collections', 'event_LOST_count', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_1h', 'rolling_std_1h', 'pct_change', 'zscore_expanding']
[INFO] Diagnóstico por sequence_length:
{
  "12": {
    "sequence_length": 12,
    "n_sequences": 6954,
    "n_positive": 2605,
    "positive_rate": 0.37460454414725336,
    "n_features": 13,
    "tensor_shape": [
      6954,
      12,
      13
    ]
  },
  "24": {
    "sequence_length": 24,
    "n_sequences": 6954,
    "n_positive": 2605,
    "positive_rate": 0.37460454414725336,
    "n_features": 13,
    "tensor_shape": [
      6954,
      24,
      13
    ]
  },
  "36": {
    "sequence_length"

## 16. Achados experimentais da etapa

### 16.1 Síntese da execução

O NB13a avaliou se um tuning compacto da LSTM seria suficiente para alterar a decisão do NB13 e substituir, no NB14, os escores produzidos pelo modelo tabular vencedor do NB11.

A execução foi concluída sem erros e preservou o escopo previamente delimitado:

- cenário `W5_K24_H12_P1_TRAIN_M2S`;
- conjunto `temporal_core`, com 13 atributos;
- três comprimentos de sequência;
- três dimensões da camada LSTM;
- três valores de dropout;
- duas taxas de aprendizagem;
- triagem com uma semente;
- confirmação com três sementes;
- `TimeSeriesSplit` como protocolo principal;
- corte temporal fixo 80/20 como análise complementar.

O resultado central foi negativo para a hipótese de promoção. A melhor LSTM ajustada obteve F1 médio de 0,5485 no `TimeSeriesSplit`, valor inferior ao resultado da LSTM do NB13 e ao resultado da Regressão Logística do NB11. Assim, os escores do NB11 permanecem como entrada preferencial do NB14.

## 17. Integridade do contrato de atributos

O preflight foi concluído antes do treinamento e confirmou:

| Verificação | Resultado |
|---|---|
| `raw_core` | 6 atributos |
| `temporal_core` | 13 atributos |
| Conjunto efetivamente utilizado | `temporal_core` |
| `event_FAIL_count` nos preditores | Não |
| Igualdade `n_failed = event_FAIL_count` | Confirmada |
| Consistência de `fail_rate` | Confirmada |
| Pares de atributos exatamente duplicados | 0 |
| Dimensão final dos tensores | 13 |
| Status da auditoria | `PASS` |

O vetor utilizado foi:

```text
fail_rate
n_events
n_failed
n_machines
n_collections
event_LOST_count
lag_1
lag_2
lag_3
rolling_mean_1h
rolling_std_1h
pct_change
zscore_expanding
```

A coluna `event_FAIL_count` permaneceu disponível somente para proveniência e não integrou os tensores da LSTM.

## 18. Diagnóstico das sequências

| Comprimento | Memória temporal | Sequências | Positivos | Taxa positiva |
|---:|---:|---:|---:|---:|
| 12 janelas | 60 min | 6.954 | 2.605 | 0,3746 |
| 24 janelas | 120 min | 6.954 | 2.605 | 0,3746 |
| 36 janelas | 180 min | 6.944 | 2.605 | 0,3751 |

O comprimento de 36 janelas reduziu em dez o número de sequências disponíveis. Essa redução decorre da exigência de uma memória temporal mais longa para formar cada amostra.

## 19. Cobertura experimental efetiva

O plano nominal foi integralmente realizado:

| Fase | Avaliações planejadas | Avaliações realizadas | Válidas | Ignoradas |
|---|---:|---:|---:|---:|
| Triagem | 270 | 270 | 270 | 0 |
| Confirmação | 45 | 45 | 45 | 0 |
| Corte fixo | 3 | 3 | 3 | 0 |
| **Total** | **318** | **318** | **318** | **0** |

Na triagem, foram avaliadas 54 configurações, uma semente e cinco folds. Na confirmação, as três melhores configurações da triagem foram reavaliadas com três sementes e cinco folds. O corte fixo utilizou a configuração selecionada e as três sementes.

## 20. Resultado da triagem

As três primeiras configurações da triagem foram:

| Sequência | Unidades | Dropout | Taxa de aprendizagem | F1 médio |
|---:|---:|---:|---:|---:|
| 36 | 16 | 0,10 | 0,001 | 0,5601 |
| 12 | 32 | 0,10 | 0,001 | 0,5582 |
| 36 | 16 | 0,30 | 0,001 | 0,5558 |

A triagem utilizou somente a semente 42. Seus resultados foram usados para reduzir o espaço experimental, não para sustentar isoladamente a decisão final.

## 21. Resultado da confirmação

Após a repetição com três sementes, a melhor configuração confirmada foi:

```text
sequence_length = 36
units = 16
dropout = 0,30
learning_rate = 0,001
```

| Métrica | Média | Desvio-padrão |
|---|---:|---:|
| F1 | 0,5485 | 0,2994 |
| Precisão | 0,4610 | 0,2828 |
| Recall | 0,7190 | 0,2879 |
| ROC-AUC | 0,6137 | 0,1663 |
| Average Precision | 0,5319 | 0,3021 |
| Brier Score | 0,2298 | 0,0445 |
| Épocas executadas | 18,47 | 8,36 |

As três configurações confirmadas ficaram muito próximas:

| Configuração | F1 médio |
|---|---:|
| L36, 16 unidades, dropout 0,30 | 0,5485 |
| L36, 16 unidades, dropout 0,10 | 0,5479 |
| L12, 32 unidades, dropout 0,10 | 0,5477 |

A diferença entre a primeira e a terceira foi inferior a 0,001. Isso indica que nenhuma das três configurações apresentou vantagem robusta dentro do espaço confirmado.

A mudança da primeira posição entre triagem e confirmação também mostra por que a seleção não deveria depender de uma única semente.

## 22. Comparação com o NB13

| Referência | F1 médio TSCV |
|---|---:|
| Melhor LSTM do NB13 | 0,5591 |
| Melhor LSTM ajustada do NB13a | 0,5485 |
| Delta NB13a − NB13 | −0,0106 |

O tuning compacto não melhorou a LSTM do NB13. A extensão da memória para 180 minutos e o ajuste do dropout não compensaram a variabilidade temporal observada.

Esse resultado fortalece a conclusão de que o limite encontrado no NB13 não decorreu apenas de uma escolha estreita de unidades ou de comprimento de sequência.

## 23. Comparação com o vencedor do NB11

| Referência | F1 médio TSCV |
|---|---:|
| Regressão Logística do NB11 | 0,5773 |
| Melhor LSTM ajustada | 0,5485 |
| Delta LSTM ajustada − NB11 | −0,0288 |

O critério de promoção exigia ganho mínimo de 0,03 sobre o NB11. Portanto, a LSTM ajustada precisaria alcançar F1 médio de pelo menos 0,6073.

O resultado de 0,5485 ficou aproximadamente 0,0588 abaixo desse patamar. A condição de promoção não foi atendida.

A decisão quantitativa permanece:

**manter os escores do NB11 como entrada do NB14.**

## 24. Leitura do corte temporal fixo

No corte 80/20, a configuração selecionada apresentou:

| Métrica | Resultado |
|---|---:|
| F1 médio | 0,8496 |
| Desvio-padrão do F1 | 0,0036 |
| Precisão média | 0,7830 |
| Recall médio | 0,9287 |
| ROC-AUC médio | 0,7472 |
| Average Precision média | 0,8639 |
| Brier Score médio | 0,1710 |
| Taxa positiva no teste | 0,7207 |

O desempenho elevado no corte fixo não altera a decisão da etapa. O período de teste possui prevalência positiva muito alta e corresponde a um trecho tardio da série.

Como nos notebooks anteriores, o corte fixo permanece uma análise complementar. A decisão de promoção é governada pelo `TimeSeriesSplit`, que expõe a variação entre períodos sucessivos.

## 25. Interpretação da variabilidade

O desvio-padrão do F1 da melhor configuração confirmada foi 0,2994, valor elevado em relação à média de 0,5485.

A repetição por sementes reduziu o risco de selecionar uma inicialização excepcional, mas não eliminou a instabilidade entre folds. A evidência acumulada no NB13 e no NB13a indica que a mudança temporal da distribuição é mais relevante do que pequenos ajustes da arquitetura recorrente.

Consequentemente, o ganho de complexidade da LSTM não foi acompanhado por ganho consistente de generalização temporal.

## 26. Interpretação dos escores

As saídas sigmoidais da LSTM são tratadas como **escores relativos de risco temporal**. O NB13a não estabeleceu calibração operacional suficiente para interpretá-las como probabilidades confiáveis de falha.

O limiar 0,50 foi utilizado para comparação experimental padronizada. A seleção de limiares de alerta e a análise de custos pertencem ao NB14.

Como a LSTM ajustada não foi promovida, `13a_lstm_tuning_scores.parquet` permanece como artefato complementar e não substitui os escores tabulares do NB11.

## 27. Artefatos produzidos e verificados

| Artefato | Finalidade |
|---|---|
| `13a_feature_duplicate_audit.csv` | Auditoria do contrato de atributos |
| `13a_lstm_tuning_stage1_results.csv` | Resultados detalhados da triagem |
| `13a_lstm_tuning_stage1_agg.csv` | Consolidação da triagem |
| `13a_lstm_tuning_stage2_results.csv` | Resultados detalhados da confirmação |
| `13a_lstm_tuning_stage2_agg.csv` | Consolidação da confirmação |
| `13a_lstm_tuning_fixed_results.csv` | Resultados do corte fixo |
| `13a_lstm_tuning_fixed_results_agg.csv` | Consolidação do corte fixo |
| `13a_lstm_tuning_scores.parquet` | Escores da configuração selecionada |
| `13a_lstm_tuning_best_config.json` | Configuração escolhida e comparadores |
| `13a_lstm_tuning_summary.json` | Síntese estruturada da etapa |
| `13a_lstm_tuning_conclusion.txt` | Registro textual produzido pelo pipeline |

As tabelas agregadas foram recomputadas a partir dos resultados detalhados. As diferenças encontradas ficaram apenas no nível de precisão numérica de ponto flutuante.

O Parquet possui estrutura binária válida, e todos os outputs declarados no resumo foram localizados.

## 28. Política editorial de figuras

O diretório `figures_nb13a_lstm_tuning` permaneceu vazio, conforme decisão editorial registrada no próprio notebook.

O NB13a não requer figura própria porque sua contribuição é uma verificação controlada da conclusão do NB13. Os resultados centrais podem ser apresentados por meio das tabelas de configuração, métricas e decisão de promoção.

## 29. Alcance da evidência

A etapa representa tuning compacto e controlado, não busca neural exaustiva. O espaço avaliado foi suficiente para testar:

- memória entre 60 e 180 minutos;
- capacidade recorrente entre 8 e 32 unidades;
- dropout entre 0,10 e 0,30;
- duas taxas de aprendizagem;
- estabilidade das melhores configurações em três sementes.

O resultado não demonstra que toda arquitetura neural seja inadequada ao problema. Demonstra que, no espaço experimental delimitado e sob avaliação temporal progressiva, o ajuste moderado da LSTM não superou o núcleo tabular.

## 30. Encaminhamento metodológico

A cadeia experimental mantém as seguintes decisões:

1. `TRAIN_M2S` permanece como cenário principal;
2. `temporal_core` permanece com 13 atributos;
3. a Regressão Logística do NB11 permanece como referência preditiva;
4. a LSTM do NB13 permanece como experimento complementar;
5. o tuning do NB13a não altera a decisão de não promoção;
6. os escores do NB11 permanecem como entrada do NB14;
7. os resultados do NB13a integram a discussão sobre complexidade, estabilidade e ganho incremental.

## 31. Conclusão da etapa

O NB13a cumpriu o objetivo de testar se uma varredura compacta de hiperparâmetros alteraria a conclusão do NB13.

A execução confirmou que:

1. as 318 avaliações planejadas foram realizadas;
2. todas as avaliações foram válidas;
3. o vetor utilizado contém 13 atributos;
4. `event_FAIL_count` não foi usado como preditor;
5. nenhuma duplicidade exata foi encontrada;
6. a melhor configuração confirmada utilizou 36 janelas, 16 unidades, dropout 0,30 e taxa de aprendizagem 0,001;
7. seu F1 médio no `TimeSeriesSplit` foi 0,5485;
8. o resultado ficou 0,0106 abaixo da melhor LSTM do NB13;
9. o resultado ficou 0,0288 abaixo da Regressão Logística do NB11;
10. o gatilho de promoção de +0,03 não foi atendido;
11. o corte fixo apresentou F1 elevado em um período de alta prevalência, mas permaneceu complementar;
12. os escores do NB11 continuam sendo a entrada preferencial do NB14;
13. todos os artefatos declarados foram materializados;
14. a ausência de figuras próprias está de acordo com a política editorial da etapa.

O tuning compacto reforça, em vez de enfraquecer, a decisão de manter o modelo tabular como núcleo preditivo do protótipo Kaggle refinado.
